In [ ]:
import pickle
import mlflow
from fastapi import FastAPI
from pydantic import BaseModel
from mlflow import MlflowClient
from dotenv import load_dotenv
import os
import pandas as pd
import xgboost as xgb

load_dotenv(override=True)  # Carga las variables del archivo .env

mlflow.set_tracking_uri("databricks")
client = MlflowClient()

EXPERIMENT_NAME = "/Users/pipochatgpt@gmail.com/Depression_Classification_prefect"

run_ = mlflow.search_runs(order_by=["metrics.f1 DESC"],
                          output_format="list",
                          filter_string="tags.candidate = 'true'",
                          experiment_names=[EXPERIMENT_NAME]
                          )[0]
run_id = run_.info.run_id


# 2. Descargar artifacts (preprocessor)
client.download_artifacts(
    run_id=run_id,
    path="preprocessor",
    dst_path="."
)

with open("preprocessor/encoder.pkl", "rb") as f_in:
    encoder = pickle.load(f_in)

with open("preprocessor/scaler.pkl", "rb") as f_in:
    scaler = pickle.load(f_in)

c:\PROYECTO\Depression_Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 3. Cargar modelo campeón del Registry
model_name = "workspace.default.DepressionClassificationPrefect"
alias = "champion"

model_uri = f"models:/{model_name}@{alias}"
model = mlflow.pyfunc.load_model(model_uri)

In [ ]:
# 4. Preprocess de entrada
def preprocess(input_data):

    df = pd.DataFrame([input_data.dict()])

    # -- aplicar encoder a "City"
    city_encoded = encoder.transform(df[["City"]])
    city_cols = encoder.get_feature_names_out(["City"])
    city_df = pd.DataFrame(city_encoded, columns=city_cols)

    df = df.drop(columns=["City"])

    # unir
    df_proc = pd.concat([df, city_df], axis=1)

    # escalado
    df_scaled = scaler.transform(df_proc)

    return df_scaled


# 5. Predict
def make_prediction(input_data):
    X = preprocess(input_data)
    pred = model.predict(X)
    return int(pred[0])


# FASTAPI
app = FastAPI()

class InputData(BaseModel):
    Gender: int
    Age: int
    AcademicPressure: float
    StudySatisfaction: float
    SleepDuration: int
    DietaryHabits: int
    Degree: int
    City: str
    FamilyHistory: int
    SuicidalThoughts: int

@app.post("/predict")
def predict_endpoint(input_data: InputData):
    result = make_prediction(input_data)
    return {"prediction": result}